# P04 e I01. Rendimiento y traza finita

P04: clase 7, 23 de octubre; I01: clase 8, 24 de octubre de 2026.
Preparado por el PhD Esteban Hernández, CyberColombia.

[Enunciado P04](../../talleres/P04.md) · [I01](../../talleres/I01.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

El viernes el profesor demuestra; el sábado los estudiantes ejecutan y entregan en clase. Ejecutar las celdas en orden con el kernel del único `.venv`. Cada lectura, consulta y escritura queda visible. No se importan scripts del curso.

## 1. Preparar y localizar los datos
Desde la terminal en `kit/`, con `.venv` activo:
```bash
python 00_datos.py --descargar pqrs nasa
python 00_datos.py --verificar pqrs nasa
```
Primero ejecutar [02 Parquet y contratos](02_Parquet_y_contratos.ipynb): allí se leen los CSV, se convierten los tipos y se escriben el CSV proyectado, el Parquet y sus particiones. Aquí se leen esos productos; no se regeneran silenciosamente. Las muestras contienen 3.000 reportes; no permiten concluir sobre rendimiento nacional.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "pandas/P04"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
entrada = KIT / "salidas/pandas/P02"
archivo_parquet = entrada / "pqrs.parquet"
archivo_csv = entrada / "pqrs_proyeccion.csv"
particiones = entrada / "pqrs_particionado"
assert archivo_parquet.is_file(), "Ejecuta el notebook 02"
assert archivo_csv.is_file() and list(particiones.glob("**/*.parquet"))
control_entrada = json.loads((entrada / "control_formatos_pandas.json").read_text())
assert control_entrada["filas"] == 3000, "Este bloque mide las muestras de P02"
columnas = ["corte_archivo", "anio", "mes_num", "afec_cod_depto"]
# Importar antes de medir: excluimos el coste de importación de las bibliotecas.
import polars as pl
import duckdb
from time import perf_counter
import platform
mediciones = []
display(pd.DataFrame([{"python": platform.python_version(), "pandas": pd.__version__,
                       "polars": pl.__version__, "duckdb": duckdb.__version__,
                       "sistema": platform.platform()}]))


## 2. Comparar la misma consulta en cinco variantes
Agrupar **todos los años** por corte, año, mes y departamento, incluyendo claves nulas, y contar filas. Cada celda mide lectura + agregación + obtención del resultado en pandas. `%%time` informa CPU y **Wall time**; `perf_counter` guarda el tiempo transcurrido del bloque para el informe. Polars incluye `to_pandas`; DuckDB incluye la apertura/cierre de conexión.

La ordenación, normalización de tipos, comprobación y visualización se realizan después y quedan fuera del intervalo registrado. Ejecutar una vuelta de calentamiento, vaciar `mediciones` con la celda indicada y realizar tres vueltas, alternando el orden de los motores. No volver a ejecutar la configuración entre vueltas. La caché del sistema puede persistir; no llamar a esto una prueba con caché fría. Este ejercicio comparte el proceso del kernel y no mide RSS ni arranque de procesos.


### pandas: lectura inmediata y groupby


In [ ]:
%%time
inicio = perf_counter()
datos_pandas = pd.read_parquet(archivo_parquet, columns=columnas)
resultado_pandas = (datos_pandas.groupby(columnas, dropna=False)
    .size().reset_index(name="reportes"))
segundos = perf_counter() - inicio
mediciones.append({"motor": "pandas", "segundos": segundos})


### Polars: lectura diferida y ejecución explícita
`scan_parquet` prepara el plan; `collect` lo ejecuta. Medir solo la creación de `consulta` no mide el procesamiento.


In [ ]:
%%time
inicio = perf_counter()
consulta = (pl.scan_parquet(archivo_parquet)
    .group_by(columnas).agg(pl.len().alias("reportes")))
resultado_polars = consulta.collect(engine="streaming").to_pandas()
segundos = perf_counter() - inicio
mediciones.append({"motor": "polars", "segundos": segundos})


### DuckDB: SQL sobre Parquet
Las tres variantes siguientes usan exactamente el mismo SQL. El CSV se lee como texto para conservar los ceros de los códigos; por ello incluye un coste distinto de interpretación del formato.


In [ ]:
sql = """
SELECT corte_archivo, anio, mes_num, afec_cod_depto, COUNT(*) AS reportes
FROM reportes
GROUP BY corte_archivo, anio, mes_num, afec_cod_depto
"""


### duckdb


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(archivo_parquet)).create_view("reportes")
    resultado_duckdb = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb", "segundos": segundos})


### duckdb_particionado


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(particiones / "**/*.parquet"), hive_partitioning=True).create_view("reportes")
    resultado_duckdb_particionado = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_particionado", "segundos": segundos})


### duckdb_csv


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_csv(str(archivo_csv), header=True, all_varchar=True).create_view("reportes")
    resultado_duckdb_csv = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_csv", "segundos": segundos})


## 3. Verificar antes de comparar tiempos
El CSV devuelve claves numéricas como texto; armonizamos tipos fuera de la medición. No basta con comparar el total: cada grupo debe coincidir. Inspeccionar primero los tipos originales. Repetir esta comprobación después de cada vuelta.


In [ ]:
resultados = {"pandas": resultado_pandas, "polars": resultado_polars,
              "duckdb": resultado_duckdb, "duckdb_particionado": resultado_duckdb_particionado,
              "duckdb_csv": resultado_duckdb_csv}
comparables = {}
for motor, resultado in resultados.items():
    print(motor, resultado.dtypes.to_dict())
    normalizado = resultado.copy()
    for columna in ["anio", "mes_num", "reportes"]:
        normalizado[columna] = pd.to_numeric(normalizado[columna], errors="raise").astype("Int64")
    normalizado["corte_archivo"] = normalizado["corte_archivo"].astype("string")
    normalizado["afec_cod_depto"] = normalizado["afec_cod_depto"].astype("string").str.zfill(2)
    comparables[motor] = normalizado.sort_values(columnas).reset_index(drop=True)
    assert normalizado["reportes"].sum() == 3000
for motor, resultado in comparables.items():
    pd.testing.assert_frame_equal(comparables["pandas"], resultado)
display(comparables["pandas"])
print("Los cinco resultados coinciden grupo por grupo")


### Excluir el calentamiento
Ejecutar la siguiente celda **una vez**, tras verificar la primera vuelta. Después repetir únicamente las cinco celdas medidas y la comprobación anterior tres veces; alternar su orden. No ejecutar de nuevo esta celda al finalizar.


In [ ]:
mediciones = []


### Guardar las tres repeticiones
Ejecutar tras completar las tres vueltas. Este archivo documenta el alcance del kernel; no sustituye las mediciones de RSS en procesos independientes.


In [ ]:
tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == 15, "Faltan las tres vueltas de cinco motores después del calentamiento"
assert tiempos.groupby("motor").size().eq(3).all()
tiempos["repeticion"] = tiempos.groupby("motor").cumcount() + 1
tiempos["alcance"] = "kernel: lectura, agregación y resultado pandas; 3000 filas"
resumen = tiempos.groupby("motor")["segundos"].agg(["median", "min", "max"]).reset_index()
tiempos.to_csv(OUT / "benchmark_notebook.csv", index=False)
resumen.to_csv(OUT / "benchmark_notebook_resumen.csv", index=False)
comparables["pandas"].to_csv(OUT / "conteos_verificados.csv", index=False)
display(resumen)


## 4. Escala real y automatización, después de comprender las consultas
[P04](../../talleres/P04.md) conserva la extensión de cinco variantes en procesos independientes, dos hilos, calentamiento, tres repeticiones y sondeo RSS cada 10 ms. Revisar `trabajador` y `benchmark` en `kit/pqrs_talleres.py`: identificar en el código cada consulta que acabas de ejecutar antes de usar la automatización.

Los tiempos del CLI incluyen arranque y exportación; **no mezclar** esas mediciones con las del kernel. El CLI escribe `benchmark_pqrs.csv` y su resumen; el notebook usa otros nombres para distinguir el alcance. Registrar máquina, versiones e hilos. Las bibliotecas del kernel pueden usar distintos números de hilos: esta comparación describe la configuración local, no aísla el efecto del motor.

El volumen completo se descarga explícitamente según P04; reservar al menos 8 GB de disco más el entorno. Con 8 GB de RAM mantener pandas en muestras. Los completos concilian 2.444.766 reportes. No cambiar la ruta de este bloque de muestras a los completos sin revisar su preparación y memoria.

**Detenerse aquí en la clase 7.** Retomar eventos en la clase 8.

## 5. Leer las observaciones NASA
`json.load` lee el documento; `DataFrame` convierte el diccionario de parámetros en filas. Convertimos las fechas y separamos los faltantes (-999), sin inventar lluvia. La fecha diaria se usa como tiempo de evento por convención didáctica.


In [ ]:
with open(RAW / "nasa.json", encoding="utf-8") as archivo:
    nasa = json.load(archivo)
lluvia = pd.Series(nasa["properties"]["parameter"]["PRECTOTCORR"], name="lluvia_mm")
observaciones = lluvia.rename_axis("event_id").reset_index()
observaciones["fecha_evento"] = pd.to_datetime(observaciones["event_id"], format="%Y%m%d")
observaciones["lluvia_mm"] = pd.to_numeric(observaciones["lluvia_mm"], errors="raise").replace(-999, float("nan"))
assert observaciones["event_id"].is_unique
display(observaciones.head(12))


## 6. Construir el orden de entrega y unir los valores reales
Primero llegan los días 1–4 y 6–12; luego se reintenta el 6 y finalmente llega el 5. `merge(validate="many_to_one")` permite entregas repetidas pero exige una observación por fecha. Inspeccionar la tabla antes de aplicar reglas.


In [ ]:
dias = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 6, 5]
entregas = pd.DataFrame({"event_id": [f"202501{dia:02d}" for dia in dias]})
entregas["orden_entrega"] = range(1, len(entregas) + 1)
entregas = entregas.merge(observaciones, on="event_id", how="left", validate="many_to_one", indicator=True)
assert entregas["_merge"].eq("both").all() and entregas["lluvia_mm"].notna().all()
display(entregas)


## 7. Recorrer la traza paso a paso
Antes de cada entrega, el límite es la fecha máxima observada menos tres días. Primero comprobamos si el identificador ya fue aceptado; después, si la fecha es anterior al límite. Una fecha igual al límite se admite. La lista `traza` conserva la decisión y el límite usado; `vistos` solo guarda aceptados. Esta regla finita no reproduce toda la semántica de Spark.


In [ ]:
vistos = set()
ultima_fecha = None
traza = []
for entrega in entregas.itertuples(index=False):
    limite = ultima_fecha - pd.Timedelta(days=3) if ultima_fecha is not None else None
    if entrega.event_id in vistos:
        estado = "duplicado"
    elif limite is not None and entrega.fecha_evento < limite:
        estado = "tardio"
    else:
        estado = "aceptado"
        vistos.add(entrega.event_id)
    traza.append({"orden_entrega": entrega.orden_entrega, "event_id": entrega.event_id,
                  "fecha_evento": entrega.fecha_evento, "lluvia_mm": entrega.lluvia_mm,
                  "watermark_antes": limite, "estado": estado})
    ultima_fecha = max(ultima_fecha, entrega.fecha_evento) if ultima_fecha is not None else entrega.fecha_evento
    print(entrega.orden_entrega, entrega.event_id, "límite:", limite, estado)
traza = pd.DataFrame(traza)


## 8. Conciliar y escribir las evidencias
Comparar entregas, fechas distintas y aceptados. Explicar qué observación falta en la suma aceptada y proponer una corrección antes de continuar con Spark en E11.


In [ ]:
aceptados = traza.loc[traza["estado"].eq("aceptado")]
control = {"entregas": len(traza), "registros_reales_distintos": int(traza["event_id"].nunique()),
           "aceptados": len(aceptados), "duplicados": int(traza["estado"].eq("duplicado").sum()),
           "tardios": int(traza["estado"].eq("tardio").sum()),
           "suma_aceptada_mm": float(aceptados["lluvia_mm"].sum()),
           "suma_fechas_unicas_mm": float(entregas.drop_duplicates("event_id")["lluvia_mm"].sum())}
assert (control["entregas"], control["registros_reales_distintos"], control["aceptados"], control["duplicados"], control["tardios"]) == (13, 12, 11, 1, 1)
assert abs(control["suma_aceptada_mm"] - 41.44) < 1e-8
traza.to_csv(OUT / "traza_eventos_nasa.csv", index=False)
with open(OUT / "control_eventos_intro.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
recuperada = pd.read_csv(OUT / "traza_eventos_nasa.csv", dtype={"event_id": "string"},
                         parse_dates=["fecha_evento", "watermark_antes"])
pd.testing.assert_frame_equal(traza.astype({"event_id": "string"}), recuperada)
display(control)
display(recuperada)


## Entrega
Adjuntar tiempos, resultados equivalentes, configuración y alcance de la medición; distinguir notebook y CLI. Entregar la traza y explicar el reintento y el atraso. Modificar una condición de consulta, anticipar el resultado y verificarlo en todos los motores. No sumar lluvia con PQRS ni fabricar tiempos de evento para reportes.
